# Plot Training Return vs Environment Step

This notebook plots `Train_EpisodeReturn` from one or more `log.csv` files.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8")

In [ ]:
# Add one or more log.csv paths relative to the hw3 directory.
log_paths = [
    "exp/LunarLander-v2_dqn_sd1_20260310_150011/log.csv",
    "exp/CartPole-v1_dqn_sd1_20260308_195315/log.csv",
]

# Optional custom labels (same order as log_paths).
labels = []

# Rolling window for smoothing (set 1 to disable smoothing).
rolling_window = 20

In [ ]:
def load_training_curve(csv_path: Path):
    df = pd.read_csv(csv_path)
    required = ["step", "Train_EpisodeReturn"]
    for c in required:
        if c not in df.columns:
            raise ValueError(f"{csv_path} is missing required column: {c}")

    train_df = df[["step", "Train_EpisodeReturn"]].dropna()
    train_df = train_df.sort_values("step")
    return train_df


base_dir = Path.cwd()
resolved_paths = [base_dir / p for p in log_paths]
for p in resolved_paths:
    if not p.exists():
        raise FileNotFoundError(f"Missing file: {p}")

print("Loaded paths:")
for p in resolved_paths:
    print(" -", p)

In [ ]:
plt.figure(figsize=(9, 5))

for i, p in enumerate(resolved_paths):
    train_df = load_training_curve(p)
    label = labels[i] if i < len(labels) else p.parent.name
    y = train_df["Train_EpisodeReturn"]
    if rolling_window > 1:
        y = y.rolling(rolling_window, min_periods=1).mean()
    plt.plot(train_df["step"], y, linewidth=2, label=label)

plt.xlabel("Environment Step")
plt.ylabel("Training Episode Return")
plt.title("Training Return vs Environment Step")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()